In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

In [ ]:
IS_SAMPLE      = False
NEGATIVE_RATIO = 4
CHUNK_SIZE     = 500_000

PROCESSED_DATA_DIR = '../data/processed'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.parquet')
FEAT_OUT   = os.path.join(PROCESSED_DATA_DIR, 'features.parquet')

In [ ]:
print("Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)
lf_cands = pl.scan_parquet(CAND_PATH)

print("Bước 2: Tính toán đặc trưng thống kê (User Stats & Item Stats)...")
user_stats = lf_train.group_by('mapped_user_id').agg([
    pl.len().alias('user_total_actions'),
    pl.col('rating').mean().alias('user_avg_rating_given')
])

item_stats = lf_train.group_by('mapped_item_id').agg([
    pl.len().alias('item_total_sales'),
    pl.col('rating').mean().alias('item_actual_avg_rating')
])

print("Bước 3: Xây dựng tập nhãn (Positives & Hard Negatives)...")
# Lấy tương tác thật (Nhãn 1)
lf_positives = lf_train.select(['mapped_user_id', 'mapped_item_id']).with_columns(pl.lit(1).alias('label').cast(pl.Int8))

# Lấy mẫu âm khó từ Candidates (Nhãn 0)
lf_hard_negatives = (
    lf_cands.select(['mapped_user_id', 'mapped_item_id'])
    .join(lf_train.select(['mapped_user_id', 'mapped_item_id']), on=['mapped_user_id', 'mapped_item_id'], how='anti')
    .group_by('mapped_user_id', maintain_order=True)
    .head(NEGATIVE_RATIO)
    .with_columns(pl.lit(0).alias('label').cast(pl.Int8))
)

# Thu thập tập nhãn bằng Streaming để giải phóng RAM
df_labels = pl.concat([lf_positives, lf_hard_negatives]).collect(streaming=True)
print(f"Tổng số mẫu huấn luyện (Dòng): {df_labels.height:,}")

print("Bước 4: Nối toàn bộ Đặc trưng và Ghi trực tiếp xuống ổ cứng (Sink Parquet)...")
# Chỉ lấy các cột rank từ bảng cands để làm feature cho XGBoost
lf_ranks = lf_cands.select(['mapped_user_id', 'mapped_item_id', 'sasrec_rank', 'lightgcn_rank'])

(
    df_labels.lazy()
    .join(lf_ranks, on=['mapped_user_id', 'mapped_item_id'], how='left')
    .join(lf_meta, on='mapped_item_id', how='left')
    .join(user_stats, on='mapped_user_id', how='left')
    .join(item_stats, on='mapped_item_id', how='left')
    .with_columns([
        # Đổi Rank thành Điểm (1/Rank). Rỗng thì được 0 điểm.
        (1.0 / pl.col('sasrec_rank')).fill_null(0.0).alias('sasrec_score'),
        (1.0 / pl.col('lightgcn_rank')).fill_null(0.0).alias('lightgcn_score'),
        
        pl.col('price').fill_null(0.0),
        pl.col('average_rating').fill_null(0.0),
        pl.col('rating_number').fill_null(0),
        pl.col('user_total_actions').fill_null(0),
        pl.col('item_total_sales').fill_null(0),
        pl.col('user_avg_rating_given').fill_null(3.0)
    ])

    .drop(['sasrec_rank', 'lightgcn_rank'])
    .sink_parquet(FEAT_OUT)
)

del df_labels, lf_train, lf_meta, lf_cands, user_stats, item_stats
gc.collect()

print(f"Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: {FEAT_OUT}")

Feature Engineering chunk loop completed.
